# Giai đoạn bổ sung — So sánh Backbone & Ablation Study (Chương 5.1.1, 5.4)

Notebook này chạy 2 việc CÓ THỂ làm ngay trên proxy data (không cần dữ liệu thật):
1. So sánh 3 backbone (MobileNetV2, ResNet18, EfficientNet-B0) — mục 5.1.1
2. Ablation freeze-strategy + augmentation on/off — mục 5.4

⚠️ Mỗi lần train tốn vài chục phút trên Colab T4 — tổng thời gian notebook này có thể 1–2 giờ tùy epoch cấu hình trong `configs/default.yaml`.

In [ ]:
# !git clone <repo-url> && %cd bitss-stool-classification
# !pip install -r requirements.txt -q

## Phần 1 — So sánh Backbone (mục 5.1.1)

Train + evaluate lần lượt từng backbone, mỗi backbone lưu report riêng theo quy ước `outputs/figures/backbone_<tên>/` để `generate_report_tables.py` đọc được.

In [ ]:
backbones = ['mobilenet_v2', 'resnet18', 'efficientnet_b0']

for bb in backbones:
    print(f'\n{"="*60}\nTraining backbone: {bb}\n{"="*60}')
    !python src/train.py --config configs/default.yaml \
        --backbone {bb} --checkpoint_dir outputs/checkpoints/backbone_{bb}
    !python src/evaluate.py --config configs/default.yaml \
        --checkpoint outputs/checkpoints/backbone_{bb}/best.pt \
        --report_dir outputs/figures/backbone_{bb}

In [ ]:
import json

for bb in backbones:
    with open(f'outputs/figures/backbone_{bb}/test_metrics.json') as f:
        m = json.load(f)
    print(f"{bb}: acc={m['accuracy']:.4f} f1_macro={m['f1_macro']:.4f} kappa={m['kappa_quadratic']:.4f}")

## Phần 2 — Ablation Study (mục 5.4)

Chạy tự động qua `src/run_ablation.py` — script tự tạo config biến thể, train, evaluate, và tổng hợp bảng kết quả.

In [ ]:
!python src/run_ablation.py --config configs/default.yaml --ablation both

In [ ]:
import pandas as pd
ablation_df = pd.read_csv('outputs/figures/ablation_results.csv')
ablation_df

## Sinh bảng Markdown tổng hợp cho báo cáo (mục 5.1.1 + 5.4)

In [ ]:
!python src/generate_report_tables.py --config configs/default.yaml

In [ ]:
with open('outputs/figures/report_tables.md', encoding='utf-8') as f:
    print(f.read())

Copy nội dung ở trên vào đúng vị trí Chương 5 của báo cáo.